In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import duckdb 
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = duckdb.query("SELECT * FROM '../outputs/training_data.parquet'").to_df()
with open("../outputs/features_final.json") as f:
    data = json.load(f)
TARGET = 'hpa_relative'
FEATURES_FINAL = [
    'zhvi_yoy_smooth',
    'zhvi_mom_3m',
    'zhvi_volatility_6m',
    'zori_yoy_smooth',
    'zori_mom_3m',
    'zori_volatility_6m',
    'unemployment_rate',
    'unemployment_3m_delta',
    'jobs_yoy_pct',
    'permits_yoy_pct',
    'mortgage_rate_3m_delta',
    'mortgage_rate_volatility_6m',
    'cpi_yoy_pct',
    'price_to_income_ratio',
    'piti_rate_pressure',
    'rent_to_income_ratio',
    'piti_shock'
]
df_clean = df[FEATURES_FINAL + [TARGET, 'city', 'month']].dropna()
df_clean = df_clean.sort_values('month').reset_index(drop=True)

print(f"Modeling rows: {len(df_clean):,}")
print(f"Features: {len(FEATURES_FINAL)}")

Modeling rows: 2,533
Features: 17


In [40]:
# Roughly 65% training data and 17.5% test and validation data split which is good since data sample size isn't huge
train = df_clean[df_clean['month'] <  '2024-01-01']
validation = df_clean[df_clean['month'] >= '2024-01-01']  # 2024+ only
#test = df_clean[df_clean['month'] >= '2024-01-01']
print(f"\nTraining dataset length = {len(train)}")
print(f"Validation dataset length = {len(validation)}")


X_train = train[FEATURES_FINAL]
y_train = train[TARGET]
X_val = validation[FEATURES_FINAL]
y_val = validation[TARGET]


Training dataset length = 2078
Validation dataset length = 455


In [41]:
# Standardization
# Fit scaler on TRAIN only — prevents leakage of val/test scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

In [42]:
print("Target distribution by period:")
print(f"  Train mean HPA:      {y_train.mean():.4f}  ({y_train.mean()*100:.1f}%)")
print(f"  Val mean HPA:        {y_val.mean():.4f}   ({y_val.mean()*100:.1f}%)")
print(f"\n  Train std:           {y_train.std():.4f}")
print(f"  Val std:             {y_val.std():.4f}")
print(f"\n  Train range: {y_train.min():.3f} to {y_train.max():.3f}")
print(f"  Val range:   {y_val.min():.3f} to {y_val.max():.3f}")

Target distribution by period:
  Train mean HPA:      0.0815  (8.2%)
  Val mean HPA:        0.0020   (0.2%)

  Train std:           0.0806
  Val std:             0.0295

  Train range: -0.140 to 0.403
  Val range:   -0.063 to 0.081


In [47]:
print("Rows with non-null hpa_12m_forward by year:")
df_clean['year'] = pd.to_datetime(df_clean['month']).dt.year
print(df_clean.groupby('year')['hpa_12m_forward'].agg(['count', 'mean', 'std']))

print("\nHow many val rows have valid (non-null) target?")
val_check = df_clean[df_clean['month'] >= '2023-01-01'].copy()
print(f"Val rows total:           {len(val_check):,}")
print(f"Val rows target not null: {val_check['hpa_12m_forward'].notna().sum():,}")
print(f"Val rows target is null:  {val_check['hpa_12m_forward'].isna().sum():,}")

# Check target distribution for rows where we DO have valid forward data
print("\nTarget stats for rows with non-null hpa_12m_forward in 2023:")
valid_2023 = df_clean[
    (df_clean['month'] >= '2023-01-01') &
    (df_clean['month'] <  '2024-01-01') &
    (df_clean['hpa_12m_forward'].notna())
]
print(f"Count: {len(valid_2023)}")
print(f"Mean:  {valid_2023['hpa_12m_forward'].mean():.4f}")
print(f"Range: {valid_2023['hpa_12m_forward'].min():.4f} to {valid_2023['hpa_12m_forward'].max():.4f}")

Rows with non-null hpa_12m_forward by year:
      count      mean       std
year                           
2019    420  0.057544  0.025772
2020    399  0.157738  0.055635
2021    420  0.156801  0.072945
2022    419  0.004273  0.050128
2023    420  0.034866  0.029497
2024    420  0.002851  0.029512
2025     35 -0.008349  0.027091

How many val rows have valid (non-null) target?
Val rows total:           875
Val rows target not null: 875
Val rows target is null:  0

Target stats for rows with non-null hpa_12m_forward in 2023:
Count: 420
Mean:  0.0349
Range: -0.0732 to 0.1226


In [ ]:
# Split: last available year as validation, everything before as train
max_month = pd.to_datetime(df_valid['month']).max()
val_start = max_month - pd.DateOffset(months=12)

train      = df_valid[pd.to_datetime(df_valid['month']) <  val_start]
validation = df_valid[pd.to_datetime(df_valid['month']) >= val_start]

print(f"Train: {len(train):,} rows | {train['month'].min()} → {train['month'].max()}")
print(f"Val:   {len(validation):,} rows | {validation['month'].min()} → {validation['month'].max()}")
print(f"\nTrain target: mean={train['hpa_12m_forward'].mean():.4f}, std={train['hpa_12m_forward'].std():.4f}")
print(f"Val target:   mean={validation['hpa_12m_forward'].mean():.4f}, std={validation['hpa_12m_forward'].std():.4f}")

In [48]:
# Step 1: Only keep rows where target is genuinely observed
df_valid = df_clean[df_clean['hpa_12m_forward'].notna()].copy()
df_valid = df_valid.sort_values('month').reset_index(drop=True)

print(f"Rows with valid target: {len(df_valid):,}")
print(f"Date range: {df_valid['month'].min()} → {df_valid['month'].max()}")
print(f"\nRows per year:")
print(df_valid.groupby(df_valid['month'].str[:4])['hpa_12m_forward'].agg(['count', 'mean']))

Rows with valid target: 2,533
Date range: 2019-01-01 00:00:00 → 2025-01-01 00:00:00

Rows per year:


AttributeError: Can only use .str accessor with string values!

In [46]:
# RidgeCV
# TimeSeriesSplit within train set only
# Searches 50 alpha values from 0.001 to 1000

tscv = TimeSeriesSplit(n_splits=5)
alphas = np.logspace(-3, 3, 50)
# Step 1: Diagnose
print("Target distribution by period:")
print(f"Train: mean={y_train.mean():.4f}, std={y_train.std():.4f}")
print(f"Val:   mean={y_val.mean():.4f}, std={y_val.std():.4f}")

# Step 2: Expand train window
train      = df_clean[df_clean['month'] <  '2024-01-01']
validation = df_clean[df_clean['month'] >= '2024-01-01']

X_train = train[FEATURES_FINAL]
y_train = train[TARGET]
X_val   = validation[FEATURES_FINAL]
y_val   = validation[TARGET]

# Refit scaler + models on expanded train
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

ridge = RidgeCV(alphas=alphas, cv=tscv, scoring='r2')
ridge.fit(X_train_s, y_train)

print(f"Ridge Val R² (expanded train): {r2_score(y_val, ridge.predict(X_val_s)):.4f}")
# This should be significantly better — likely 0.15–0.35

Target distribution by period:
Train: mean=0.0815, std=0.0806
Val:   mean=0.0020, std=0.0295
Ridge Val R² (expanded train): -3.8079


In [ ]:
# ElasticNetCV

enet = ElasticNetCV(
    l1_ratio=[0.1, 0.2, 0.3, 0.5, 0.7, 0.9],
    alphas=np.logspace(-3, 2, 30),
    cv=tscv,
    max_iter=10000,
    random_state=42
)
enet.fit(X_train_s, y_train)

enet_train_r2 = r2_score(y_train, enet.predict(X_train_s))
enet_val_r2 = r2_score(y_val,   enet.predict(X_val_s))
enet_val_mae = mean_absolute_error(y_val, enet.predict(X_val_s))

print(f"\nElasticNet — best alpha: {enet.alpha_:.4f}")
print(f"ElasticNet — best l1_ratio: {enet.l1_ratio_:.2f}")
print(f"ElasticNet — Train R²: {enet_train_r2:.4f}")
print(f"ElasticNet — Val R²: {enet_val_r2:.4f}")
print(f"ElasticNet — Val MAE: {enet_val_mae:.4f}")

# Check which features ElasticNet zeroed out
enet_coefs = pd.DataFrame({
    'feature': FEATURES_FINAL,
    'enet_coef': enet.coef_
})
zeroed = enet_coefs[enet_coefs['enet_coef'] == 0]['feature'].tolist()
print(f"\nElasticNet zeroed out: {zeroed}")


ElasticNet — best alpha: 0.0073
ElasticNet — best l1_ratio: 0.90
ElasticNet — Train R²: 0.6035
ElasticNet — Val R²: -1.3152
ElasticNet — Test R²: -0.6973
ElasticNet — Val MAE: 0.0313

ElasticNet zeroed out: ['zhvi_yoy_smooth', 'zhvi_volatility_6m', 'zori_yoy_smooth', 'jobs_yoy_pct', 'mortgage_rate_3m_delta', 'mortgage_rate_volatility_6m', 'price_to_income_ratio', 'rent_to_income_ratio', 'piti_shock']


In [29]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_s, y_train)

rf_val_r2  = r2_score(y_val,  rf.predict(X_val_s))
rf_test_r2 = r2_score(y_test, rf.predict(X_test_s))

print(f"\nRandom Forest — Val R²:  {rf_val_r2:.4f}")
print(f"Random Forest — Test R²: {rf_test_r2:.4f}")
print(f"Gap vs Ridge (val):      {rf_val_r2 - ridge_val_r2:.4f}")
print("(Gap = nonlinear signal Ridge can't capture)")


Random Forest — Val R²:  -0.4191
Random Forest — Test R²: -6.1755
Gap vs Ridge (val):      0.2458
(Gap = nonlinear signal Ridge can't capture)
